# 03 — X-CLIP text-conditioned zero-shot on Read My Ears

**Goal:** check whether text-video CLIP produces signal **without ANY classifier training** (unlike V-JEPA-2 + linear probe, which required training on embeddings).

**Model:** `microsoft/xclip-base-patch16-16-frames` — text-video CLIP, video and text embedded in a shared space. Per clip: similarity to a set of prompts → pred = argmax.

**Test 3 prompt strategies:**
1. Short binary: `'horse moving its ears'` vs `'calm horse'`
2. Cinematic: `'a video of a horse actively moving its ears'` vs `'a video of a horse holding its ears still'`
3. Multi-prompt voting: several variants per class, mean similarity

**Strategic question:** does a zero-shot text-conditioned approach work for the ear-movement task? If yes — a fundamental advantage for scaling to 24 RHpE behaviors (24 prompts instead of 24 trained classifiers).


## 1. Setup

## 0. Pre-import monkey patch (transformers 5.7.0 / tokenizers 0.23 compat)

In [ ]:
# Monkey patch dla bug w transformers 5.7.0 + tokenizers 0.23.0rc0:
# CLIPTokenizer calls processors.RobertaProcessing(cls=...) but the new API requires cls_token=.
# Bez tego patch'a XCLIPProcessor.from_pretrained padnie z TypeError.
from tokenizers import processors
_orig_RP = processors.RobertaProcessing

class _PatchedRobertaProcessing:
    def __new__(cls_meta, **kwargs):
        if 'cls' in kwargs:
            kwargs['cls_token'] = kwargs.pop('cls')
        return _orig_RP(**kwargs)

processors.RobertaProcessing = _PatchedRobertaProcessing
print('Patch applied: processors.RobertaProcessing accepts both cls and cls_token kwargs')

In [ ]:
import os, sys, time
from pathlib import Path
import torch, cv2, numpy as np, pandas as pd

POC = Path(os.getcwd()).resolve()
if POC.name == 'notebooks':
    POC = POC.parent
os.chdir(POC)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('Device:', DEVICE)

DATA = POC / 'vendor/ReadMyEars_Dataset/data'
test_df = pd.read_csv(DATA / 'test.csv')
available = test_df[test_df['video'].apply(lambda v: (DATA / v).exists())]
print(f'Test split: {len(test_df)} clips, {len(available)} available')

## 2. Load X-CLIP base 16-frames

In [ ]:
from transformers import XCLIPProcessor, XCLIPModel

MODEL_ID = 'microsoft/xclip-base-patch16-16-frames'
t0 = time.time()
processor = XCLIPProcessor.from_pretrained(MODEL_ID)
model = XCLIPModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(f'Loaded in {time.time()-t0:.1f}s')
n_params = sum(p.numel() for p in model.parameters())
print(f'Parametry: {n_params/1e6:.0f}M')

## 3. Helper: clip → 16 frames → text similarities

In [ ]:
def read_clip_frames(clip_path, num_frames=16):
    cap = cv2.VideoCapture(str(clip_path))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n_total < 1:
        cap.release()
        return None
    indices = np.linspace(0, n_total - 1, num_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, f = cap.read()
        if not ok:
            f = frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8)
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(frames)

@torch.no_grad()
def clip_text_similarity(clip_path, prompts, num_frames=16):
    """Returns an array of logits per prompt — higher = more similar."""
    frames = read_clip_frames(clip_path, num_frames=num_frames)
    if frames is None:
        return None
    inputs = processor(
        text=prompts,
        videos=list(frames),
        return_tensors='pt',
        padding=True,
    ).to(DEVICE)
    outputs = model(**inputs)
    # logits_per_video: (1, num_prompts) — higher = more similar
    return outputs.logits_per_video.squeeze(0).cpu().float().numpy()

# Smoke test
sample_clip = DATA / 'videos/action_S1.mp4_0_.mp4'
test_prompts = ['horse moving its ears', 'calm horse']
t0 = time.time()
sims = clip_text_similarity(sample_clip, test_prompts)
print(f'Similarity per prompt: {sims}, czas: {time.time()-t0:.2f}s')
print(f'Pred: {test_prompts[int(np.argmax(sims))]}')

## 4. Eval — strategy 1: short binary prompts

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def eval_strategy(prompts_action, prompts_background, test_df_subset, num_frames=16):
    """Per klip: mean similarity dla action prompts vs background prompts."""
    all_prompts = prompts_action + prompts_background
    n_action = len(prompts_action)
    rows = []
    t0 = time.time()
    for _, row in test_df_subset.iterrows():
        clip = DATA / row['video']
        sims = clip_text_similarity(clip, all_prompts, num_frames=num_frames)
        if sims is None:
            continue
        action_score = sims[:n_action].mean()
        bg_score = sims[n_action:].mean()
        pred = 1 if action_score > bg_score else 0
        gt = 1 if row['label'] == 'action' else 0
        rows.append({'clip': row['video'].split('/')[-1], 'gt': gt, 'pred': pred,
                     'action_score': float(action_score), 'bg_score': float(bg_score)})
    df = pd.DataFrame(rows)
    print(f'  Eval w {time.time()-t0:.0f}s na {len(df)} klipach')
    y_te = df['gt'].values
    y_pred = df['pred'].values
    return {
        'accuracy': accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'recall': recall_score(y_te, y_pred),
        'f1': f1_score(y_te, y_pred),
        'cm': confusion_matrix(y_te, y_pred).tolist(),
        'df': df,
    }

print('=== Strategy 1: short binary prompts ===')
s1 = eval_strategy(
    prompts_action=['horse moving its ears'],
    prompts_background=['calm horse'],
    test_df_subset=available,
)
for k in ['accuracy', 'precision', 'recall', 'f1']:
    print(f'  {k:10s} {s1[k]:.3f}')
print(f'  cm: {s1["cm"]}')

## 5. Eval — strategy 2: cinematic prompts

In [ ]:
print('=== Strategia 2: cinematic prompts ===')
s2 = eval_strategy(
    prompts_action=['a video of a horse actively moving its ears'],
    prompts_background=['a video of a horse holding its ears still'],
    test_df_subset=available,
)
for k in ['accuracy', 'precision', 'recall', 'f1']:
    print(f'  {k:10s} {s2[k]:.3f}')
print(f'  cm: {s2["cm"]}')

## 6. Eval — strategy 3: multi-prompt voting

In [ ]:
print('=== Strategia 3: multi-prompt voting ===')
s3 = eval_strategy(
    prompts_action=[
        'horse moving its ears',
        'a video of a horse twitching its ears',
        'horse rotating its ears',
        'a horse with active ear movement',
    ],
    prompts_background=[
        'calm horse',
        'a horse standing still',
        'a horse with ears held steady',
        'a still horse',
    ],
    test_df_subset=available,
)
for k in ['accuracy', 'precision', 'recall', 'f1']:
    print(f'  {k:10s} {s3[k]:.3f}')
print(f'  cm: {s3["cm"]}')

## 7. Comparison with V-JEPA-2, Stage A, the paper

In [ ]:
import json, matplotlib.pyplot as plt

vjepa_results = json.load(open(POC / 'outputs/vjepa2_results.json'))
etap_a = json.load(open(POC / 'outputs/movement_detection_results.json'))

comparison = {
    'Paper claim (Alves 2025)': 0.875,
    'V-JEPA-2 + linear probe': vjepa_results['best_linear_probe']['acc'],
    'V-JEPA-2 + k-NN (k=15)': vjepa_results['best_knn']['acc'],
    'X-CLIP zerolot S3 (multi-prompt)': s3['accuracy'],
    'X-CLIP zerolot S2 (cinematic)': s2['accuracy'],
    'X-CLIP zerolot S1 (binary)': s1['accuracy'],
    'Stage A movement-detection': etap_a['accuracy'],
}
print('=== Final comparison ===')
for k, v in comparison.items():
    print(f'  {k:40s} {v:.3f}')

fig, ax = plt.subplots(figsize=(10, 5))
names = list(comparison.keys())
vals = list(comparison.values())
colors = ['#888', '#2ca02c', '#1f77b4', '#9467bd', '#9467bd', '#9467bd', '#d62728']
bars = ax.barh(names, vals, color=colors)
ax.axvline(0.5, color='red', linestyle=':', alpha=0.5, label='random (50%)')
ax.set_xlabel('Accuracy on test split (n=48)')
ax.set_xlim(0, 1)
ax.set_title('Read My Ears — final approach comparison')
for bar, val in zip(bars, vals):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)
ax.legend(loc='lower right')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(POC / 'outputs/final_comparison.png', dpi=120)
plt.show()

xclip_results = {
    'model': MODEL_ID,
    'strategy_1_binary': {k: float(v) if not isinstance(v, list) else v for k, v in s1.items() if k != 'df'},
    'strategy_2_cinematic': {k: float(v) if not isinstance(v, list) else v for k, v in s2.items() if k != 'df'},
    'strategy_3_multi_prompt': {k: float(v) if not isinstance(v, list) else v for k, v in s3.items() if k != 'df'},
    'comparison': {k: float(v) for k, v in comparison.items()},
}
with open(POC / 'outputs/xclip_results.json', 'w') as f:
    json.dump(xclip_results, f, indent=2)
print(f'\nZapisano: outputs/xclip_results.json + outputs/final_comparison.png')

## 8. Strategic note

**What the X-CLIP results say about scaling to 24 RHpE behaviors:**

- **acc ≥ 0.75 (S3 multi-prompt)** → text-conditioned zero-shot is a real pivot. 24 RHpE behaviors → 24 prompts + multi-label scoring with NO training. Drastically lowers the barrier for Phase 2/3.
- **acc 0.6-0.75** → zero-shot shows signal but the noise is too high. Better strategy: V-JEPA-2 embeddings + 24 linear probes (each ~1 s of CPU training).
- **acc <0.6** → the text-conditioned approach doesn't fit this subtle task (ear movement is a very niche pattern; X-CLIP was trained on coarse YouTube actions).

X-CLIP as a proof-of-concept for 24 RHpE behaviors: if it works on ear movement (the most obvious case), the heavier RHpE behaviors (head position, mouth open, eye expression) should fare even better — they are less subtle.
